In [24]:
import numpy as np  # numpy 패키지 가져오기
import pandas as pd # pandas 패키지 가져오기
import matplotlib.pyplot as plt # 시각화 패키지 가져오기
import seaborn as sns # 시각화

## 2.데이터 전처리
import re                             # 정규식 모듈 임포트

## 3.형태소 처리
from konlpy.tag import Okt
import nltk
from collections import Counter

## 입찰결과 원본 파일에서 물품만 따로 분리 하기

In [11]:
IRP_df = pd.read_csv('C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/방위사업청_국내조달 경쟁 입찰결과_20241031.csv', encoding="cp949")
IRP_df.head()

,입찰공고번호,입찰공고차수,입찰공고명,업무구분명,계약체결형태명,계약체결방법명,낙찰자결정방법명,적격심사여부,공고기관명,공고기관코드,...,개찰시각,개찰결과구분명,최종낙찰금액,최종낙찰율,최종낙찰일자,최종낙찰업체명,최종낙찰업체대표자명,최종낙찰업체담당자명,최종낙찰업체사업자등록번호,최종낙찰업체주소
0,MHD0102,4,상륙기동헬기 조종시뮬레이터 선행연구,용역,총액계약,협상에의한계약(서류),협상,N,방위사업청(운영지원과),1690000,...,14:00:00,개찰완료,28000000.0,93.333,2016-02-11,재단법인 한국군사문제연구원,김형철,NaN,126-82-20298,"경기도 성남시 수정구 위례대로83(창곡동, 국방문화센터)"
1,MHD0130,2,특수작전용 유탄발사기사업 선행연구,용역,총액계약,협상에의한계약(서류),협상,N,방위사업청(운영지원과),1690000,...,14:00:00,개찰완료,71000000.0,95.826,2016-01-06,사단법인 안보경영연구원,전건욱,NaN,120-82-07415,"서울특별시 강남구 언주로527, 10층 (역삼동)"
2,UMM0003,2,00부대 물절약 투자대행사업(WASCO 15-A047),용역,총액계약,협상에의한계약(전자),협상,N,국군재정관리단,ZD00303,...,11:00:00,개찰완료,535467000.0,80.279,2016-01-29,주식회사 도화엔지니어링,손영일,유근영,211-81-08009,서울특별시 강남구 삼성로 438(대치동)
3,UMM0004,2,00부대 물절약 투자대행사업(WASCO 15-A048),물품,총액계약,협상에의한계약(전자),협상,N,국군재정관리단,ZD00303,...,11:00:00,유찰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,UMM0860,2,16년 음향 영상장비 유지보수 용역,용역,총액계약,일반경쟁,적격심사제,Y,국군재정관리단,ZD00303,...,11:00:00,개찰완료,396923000.0,86.972,2016-01-05,주식회사 케이원,송관섭,전대석,107-81-54150,"서울특별시 금천구 가산디지털2로14, 14층 1405, 1406호(가산동)"


In [12]:
IRP_df = IRP_df[IRP_df['업무구분명'] != '용역']

In [21]:
IRP_df.head()

,입찰공고번호,입찰공고차수,입찰공고명,업무구분명,계약체결형태명,계약체결방법명,낙찰자결정방법명,적격심사여부,공고기관명,공고기관코드,...,개찰시각,개찰결과구분명,최종낙찰금액,최종낙찰율,최종낙찰일자,최종낙찰업체명,최종낙찰업체대표자명,최종낙찰업체담당자명,최종낙찰업체사업자등록번호,최종낙찰업체주소
3,UMM0004,2,00부대 물절약 투자대행사업(WASCO 15-A048),물품,총액계약,협상에의한계약(전자),협상,N,국군재정관리단,ZD00303,...,11:00:00,유찰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,EGG0109,2,155밀리 모의신관류,물품,총액계약,일반경쟁,적격심사제,Y,방위사업청,1690000,...,14:00:00,유찰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,EGG0109,3,155밀리 모의신관류,물품,총액계약,일반경쟁,적격심사제,Y,방위사업청,1690000,...,14:00:00,유찰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,EGG0109,4,155밀리 모의신관류,물품,총액계약,일반경쟁,적격심사제,Y,방위사업청,1690000,...,14:00:00,순위확정,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,UMM0014,1,스펙트럼용 분석기 구매,물품,총액계약,제한경쟁,적격심사제,Y,국군재정관리단,ZD00303,...,11:00:00,개찰완료,71225000.0,88.021,2016-01-29,미래상사,김정윤,김정윤,301-29-35140,충청북도 청주시 상당구 월평로250번길6-0 (용암동)


#### 입찰결과_물품 분리한거 csv로 저장

In [25]:
# file_name = '입찰결과_물품.csv'
# IRP_df.to_csv(file_name, index=False, encoding = 'cp949')

## 물품데이터에서 입찰공고명만 가져오기

In [26]:
IRP_df.columns

Index(['입찰공고번호', '입찰공고차수', '입찰공고명', '업무구분명', '계약체결형태명', '계약체결방법명', '낙찰자결정방법명',
       '적격심사여부', '공고기관명', '공고기관코드', '수요기관명', '수요기관코드', '낙찰하한율', '예정가격', '기초금액',
       '추정가격', '개찰일자', '개찰시각', '개찰결과구분명', '최종낙찰금액', '최종낙찰율', '최종낙찰일자',
       '최종낙찰업체명', '최종낙찰업체대표자명', '최종낙찰업체담당자명', '최종낙찰업체사업자등록번호', '최종낙찰업체주소'],
      dtype='object')

In [34]:
IRP_T_df = IRP_df.drop(['입찰공고번호', '입찰공고차수', '업무구분명', '계약체결형태명', '계약체결방법명', '낙찰자결정방법명','적격심사여부', '공고기관명', '공고기관코드', '수요기관명', '수요기관코드', '낙찰하한율', '예정가격', '기초금액', '추정가격', '개찰일자', '개찰시각', '개찰결과구분명', '최종낙찰금액', '최종낙찰율', '최종낙찰일자','최종낙찰업체명', '최종낙찰업체대표자명', '최종낙찰업체담당자명', '최종낙찰업체사업자등록번호', '최종낙찰업체주소'], axis=1)
IRP_T_df.head()

,입찰공고명
3,00부대 물절약 투자대행사업(WASCO 15-A048)
9,155밀리 모의신관류
10,155밀리 모의신관류
11,155밀리 모의신관류
13,스펙트럼용 분석기 구매


In [ ]:
#물품_입찰결과 갯수 
len(IRP_T_df)

61432

In [ ]:
# #입찰결과_물품_입찰공고명 .csv
# file_name = '입찰결과_물품_입찰공고명.csv'
# IRP_T_df.to_csv(file_name, index=False, encoding='cp949')

In [36]:
# 1. 7종과 9종에 해당하는 핵심 키워드를 리스트로 정의합니다.
# 이 키워드들은 [별표 2] 군수품 종별 세부 분류표를 참고하여 만들었습니다.
# 필요에 따라 키워드를 더 추가하거나 수정하여 정확도를 높일 수 있습니다.
keywords_7_9 = [
    # --- 7종 (항공) ---
    '항공', '전투임무기', '공중기동기', '헬기', '감시통제기', '훈련기', 'Lynx',
    
    # --- 9종 (기동/화력/통신/전자/정비) ---
    '기동', '화력', '통신', '전자', '정비', '전차', '장갑차', '차량', '트레일러', 
    '화기', '화포', '함포', '병기', '사격', '폭발물', '무기', '전산', '감시', 
    '계측', '부품', '케이블', '굴착기', '청소기'
]

# 2. 입찰공고명을 분류하는 함수를 만듭니다.
def classify_item(title):
    """
    입찰공고명(title)에 keywords_7_9의 키워드가 하나라도 포함되어 있으면 '7종/9종'으로,
    아니면 '그 외'로 분류합니다.
    """
    # any() 함수는 리스트의 요소 중 하나라도 참이면 True를 반환합니다.
    if any(keyword in str(title) for keyword in keywords_7_9):
        return '7종/9종'
    else:
        return '그 외'

# 3. IRP_T_df에 '분류'라는 새로운 열을 만들고, classify_item 함수를 적용합니다.
# DataFrame의 '입찰공고명' 열에 있는 각 항목에 대해 함수가 실행됩니다.
IRP_T_df['분류'] = IRP_T_df.apply(classify_item)

# 4. 분류 결과를 확인합니다.
print("✅ 분류 작업 완료! 상위 5개 결과:")
print(IRP_T_df.head())

print("\n----------------------------------------\n")

# 5. 전체적인 분류 현황을 확인합니다. (각 분류별 개수)
print("📊 전체 분류 현황:")
print(IRP_T_df['분류'].value_counts())

✅ 분류 작업 완료! 상위 5개 결과:
                              입찰공고명   분류
3   00부대 물절약 투자대행사업(WASCO  15-A048)  NaN
9                       155밀리 모의신관류  NaN
10                      155밀리 모의신관류  NaN
11                      155밀리 모의신관류  NaN
13                     스펙트럼용 분석기 구매  NaN

----------------------------------------

📊 전체 분류 현황:
Series([], Name: count, dtype: int64)
